## Load and Chunk data

In [1]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("Wix/WixQA", "wixqa_expertwritten", split="train")
corpus = load_dataset("Wix/WixQA", "wix_kb_corpus", split="train")
simulated = load_dataset("Wix/WixQA", "wixqa_simulated", split="train")

In [2]:
corpus[0]

{'id': '860475a2cbc65c226ecf08729d0430584e46f35f23a3e10bb5500c2c4ad09168',
 'url': 'https://support.wix.com/en/article/wix-events-about-the-event-details-and-registration-form-pages',
 'contents': 'Wix Events: About the Event Details and Registration Form Pages\nGuests visiting your site view the events you offer on the Events List page. From there they can learn more on the Events Details page (if enabled) and \xa0complete the booking on the Registration Form Page.\xa0You can customize how these pages look to suit your events.Tips:\nCustomizations you make to the Events Details page and the Registration Form page (e.g. changing colors and fonts, hiding page elements) affect these page for all events. \xa0 If your site only has events without tickets, you have the option of disabling the Event Details Page.Events Details PageSite visitors who click to register for an event are first directed to the Event Details page (unless you hid the page). There, guests can receive more complete in

In [ ]:
from typing import List, Dict, Any, Iterable
from langchain_text_splitters import RecursiveCharacterTextSplitter


def load_wix_corpus() -> Iterable[Dict[str, Any]]:
    """
    Tải WixQA corpus (wix_kb_corpus) chỉ lấy các field cần thiết.
    """
    ds = load_dataset("Wix/WixQA", "wix_kb_corpus", split="train")
    for rec in ds:
        yield {
            "id": rec["id"],
            "title": rec.get("title"),
            "url": rec.get("url"),
            "article_type": rec.get("article_type"),
            "contents": rec.get("contents") or "",
        }


# Define các hàm split khác nhau
def split_recursive(text: str, chunk_size: int, chunk_overlap: int) -> List[int]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    return splitter.split_text(text or "")

In [ ]:
# Code pipeline chunk chính
def make_chunks_for_records(
    rec: Dict[str, any],
    strategy: str,
    chunk_size: int,
    chunk_overlap: int,
) -> List[Dict[str, any]]:
    text = rec.get("contents") or ""
    if not text.strip():
        return []

    if strategy == "recursive":
        parts = split_recursive(text, chunk_size, chunk_overlap)
    elif strategy == "token":
        # parts = split_token(text, chunk_size, chunk_overlap)
        pass
    elif strategy == "semantic":
        # thông số semantic có thể chỉnh trong hàm; chunk_size/overlap dùng cho nén hậu kì
        # parts = split_semantic(text, max_chunk_size=chunk_size)
        pass
    elif strategy == "hybrid_para_token":
        # parts = split_hybrid_para_token(text, chunk_size, chunk_overlap)
        pass
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    out = []
    for i, p in enumerate(parts):
        if not p.strip():
            continue
        out.append(
            {
                "chunk_id": f"{rec['id']}::{i}",
                "article_id": rec["id"],
                "title": rec.get("title"),
                "url": rec.get("url"),
                "article_type": rec.get("article_type"),
                "text": p,
                # "n_tokens": count_tokens(p),
                "position": i,
            }
        )
    return out

In [ ]:
import argparse
import json
import os


ap = argparse.ArgumentParser(
    description="Chunk WixQA corpus.contents với LangChain nhiều chiến lược"
)
ap.add_argument(
    "--strategy",
    type=str,
    default="token",
    choices=["recursive", "token", "semantic", "hybrid_para_token"],
    help="Chiến lược chunk",
)
ap.add_argument("--chunk_size", type=int, default=380, help="Kích thước chunk (token)")
ap.add_argument("--overlap", type=int, default=50, help="Số token overlap")
ap.add_argument(
    "--max_records", type=int, default=0, help="Giới hạn số bài (0 = toàn bộ)"
)
ap.add_argument(
    "--out", type=str, default="wix_chunks.jsonl", help="Đường dẫn file JSONL output"
)
args = ap.parse_args()

os.makedirs(os.path.dirname(args.out) or ".", exist_ok=True)

n = 0
total_chunks = 0


with open(args.out, "w", encoding="utf-8") as f:
    for rec in load_wix_corpus():
        chunks = make_chunks_for_record(
            rec,
            strategy=args.strategy,
            chunk_size=args.chunk_size,
            chunk_overlap=args.overlap,
        )
        for ch in chunks:
            f.write(json.dumps(ch, ensure_ascii=False) + "\n")
        total_chunks += len(chunks)
        n += 1
        if args.max_records and n >= args.max_records:
            break

print(f"Done. Articles processed: {n} | Chunks written: {total_chunks} → {args.out}")


## Create embedding and vectorstore

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [ ]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="demo_collection",
    vectors_config=VectorParams(size=3072, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="demo_collection",
    embedding=embeddings,
)